In [0]:
%sql
create connection if not exists yt_databricks_earthquake_conn
type HTTP
-- options (
--   url "https://earthquake.usgs.gov/fdsnws/event/1/query.geojson",
--   method "POST",
--   headers "Content-Type: application/x-www-form-urlencoded"
-- )
options(
  host = "https://earthquake.usgs.gov",
  port = 443,
  base_path = "/earthquakes/feed/v1.0/",
  bearer_token = "na"
)


In [0]:
# %sql
# use catalog yt_dev;
# use schema bronze_dev;

# select current_catalog(), current_schema();

# create volume if not exists earthquake_volume;

In [0]:
dbutils.widgets.text("catalog_name", "yt_dev", 'yt_dev')
dbutils.widgets.text("schema_name", "bronze_dev", 'bronze_dev')

current_catalog = dbutils.widgets.get("catalog_name")
current_schema = dbutils.widgets.get("schema_name")

assert current_catalog, "catalog_name is empty"
assert current_schema, "schema_name is empty"

volume_name = f"{current_catalog}.{current_schema}.earthquake_volume"

try:
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {volume_name}")
except Exception as e:
    raise RuntimeError(f"Volume creation failed due to {e}") from e


In [0]:
from databricks.sdk import WorkspaceClient

workspace = WorkspaceClient()

conn = workspace.connections.get(name= "yt_databricks_earthquake_conn")
# print(conn)
base_url = f"{conn.options['host']}{conn.options['base_path']}"
print(base_url)

In [0]:
import requests
import json
import datetime

# url = 'https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.geojson'

url = f"{base_url}summary/all_day.geojson"

try:
    response = requests.get(url=url, timeout=10)
    response.raise_for_status()
    data = response.json()
except requests.exceptions.RequestException as e:
    raise RuntimeError(f"API request failed: {e}") from e
except ValueError as e:
    raise RuntimeError(f"Invalid JSON response: {e}") from e

current_date = datetime.datetime.now().strftime("%Y%m%d")
earthquake_volume = f"/Volumes/{current_catalog}/{current_schema}/earthquake_volume/earthquake_data_{current_date}.json"

try:
    dbutils.fs.put(earthquake_volume, json.dumps(data), overwrite=True)
except Exception as e:
    raise RuntimeError(f"File write failed: {e}") from e